# Augplot: start with auto, then add the hard part in one sentence

Two examples where a short refinement replaces substantial plotting code: highlight CV winners with fold variability, then turn model outputs into a forecast diagnostic. All data here is synthetic.

**Setup:** install using the [README](../README.md#install), select your Augplot kernel, and run the first cell. It selects `openai/gpt-5.6-terra` and asks for your API key with hidden input.

First run: four generations, plus one if you enable Plotly; each may need one repair request. Identical reruns use saved code. Your provider receives a data profile; generated Python runs locally and is not sandboxed.

In [ ]:
import os
from getpass import getpass

os.environ["AUGPLOT_MODEL"] = "openai/gpt-5.6-terra"
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

In [ ]:
import augplot as ap

## 1. Start from an automatic CV comparison

Five classifiers, five CV folds, and two higher-is-better metrics. Pass the results directly to `ap.plot()` and let Augplot choose the first chart.

The next cell adds the analysis-aware details that usually take most of the plotting code.

In [ ]:
cv_results = ap.load_dataset("cv_results")
viz = ap.plot(cv_results, backend="matplotlib")

In [ ]:
print(viz.explanation)
print("Reused saved code:", viz.cache_hit)

## 2. Add the hard part in one sentence

This refinement has to aggregate folds, find a different winner for each metric, preserve variability, style individual bars, place labels above error bars, and update the legend. “Best” means highest observed mean, not statistical significance.

In [ ]:
viz.refine("Highlight the best model for each metric and show fold variability.")

In [ ]:
# The generated function and its saved version are available for inspection.
print("Saved source:", viz.history_path)
# print(viz.code)

## 3. New results, same function

These synthetic updated scores make the neural network the accuracy winner. `render()` should move the highlight automatically, without calling the LLM. Export the current function under a readable name when you're happy with it.

In [ ]:
updated_results = cv_results.copy()
is_neural_net = updated_results["model"] == "Neural network"
updated_results.loc[is_neural_net, "accuracy"] = [0.947, 0.943, 0.952, 0.938, 0.950]
viz.render(updated_results, title="Updated CV results — a new accuracy winner")

In [ ]:
from pathlib import Path

# Choose a fresh filename on reruns so we do not overwrite existing code.
export_path = Path("vis_utils.py")
version = 2
while export_path.exists():
    export_path = Path(f"vis_utils_{version}.py")
    version += 1

viz.save(export_path, function_name="plot_cv_results")

In [ ]:
# The printed snippet works for normal imports. Here we load the chosen file
# directly so this cell also works when a rerun selected a numbered filename.
import importlib.util

spec = importlib.util.spec_from_file_location(export_path.stem, export_path)
vis_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vis_utils)

fig = vis_utils.plot_cv_results(updated_results, figsize=(11, 5))
fig

## 4. Start from observed model-serving latency

Suppose an upstream model forecasts weekly inference latency. The table contains observed latency, supplied forecasts, and interval bounds. Six forecast weeks now have observations; the last two are still in the future.

Start with the observations. Augplot does not fit the forecasting model or calculate the interval.

In [ ]:
forecast_results = ap.load_dataset("forecast_results")
forecast_viz = ap.plot(
    forecast_results,
    backend="seaborn",
    prompt="Plot observed inference latency over time.",
)

## 5. Turn it into a forecast diagnostic in one sentence

A useful diagnostic needs several coordinated layers: forecast, interval band, forecast boundary, missing future observations, and interval misses. The prompt stays short because those details are already present in the data.

In [ ]:
forecast_viz.refine(
    "Overlay the forecast and interval, mark where forecasting starts, "
    "and highlight interval misses."
)

## 6. Optional: make the diagnostic interactive

Set the toggle to `True` to generate the same diagnostic with Plotly hover details.

In [ ]:
RUN_PLOTLY = False

if RUN_PLOTLY:
    interactive_viz = ap.plot(
        forecast_results,
        prompt="Show the forecast diagnostic with hover details for every model output.",
        backend="plotly",
    )

**Rerunning:** run from the original `ap.plot()` cell to replay the same refinement. Repeating only `refine()` edits the current version again. Keep `.augplot/` with this notebook. [How history works](../docs/visualization-history.md).